## Setup & Imports

In [ ]:
import sys
import yaml
import shutil
import random
from pathlib import Path

current_dir = Path.cwd()
src_dir = current_dir.parent / "scripts"
if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

try:
    from create_patches import create_training_patches
    from repo_paths import get_repo_paths
    from data_generator import generate_synthetic_data
    from create_splits import split_dataset
except ImportError as e:
    print(f"Import Error: {e}")

In [8]:
# Get all the relevant paths
repo_paths = get_repo_paths()

# Get the path to the config file
config_path = repo_paths["config_dir"] / "dataset_config.yaml"

if not config_path.exists():
    raise FileNotFoundError(f"Config file not found at: {config_path}")

# Read the config file into a dictionary
with open(config_path, "r") as f:
    config = yaml.safe_load(f)


## Create Patches

In [3]:
stats = create_training_patches(config=config)

print("\n--- Processing Complete ---")
print(f"Positives: {stats['pos']}")
print(f"Hard Negatives: {stats['neg_hard']}")
print(f"Backgrounds (Textured): {stats['neg_textured']}")
print(f"Backgrounds (Flat): {stats['neg_flat']}")
print(f"Discarded (Ambiguous): {stats['discarded_ambiguous']}")
print(f"Skipped (Exact Duplicates): {stats['duplicates_skipped']}")
print(f"Skipped (Perceptual Duplicates): {stats['skipped_perceptual_dup']}")

Pre-scanning output directory for existing patches...
  Found 0 existing unique patches (MD5).
Starting Patch Generation...
  Input: /home/nico/workspace/github.com/Nico-Sander/qr-code-detection-nico/dataset/full_sized/images
  Images Found: 3041


Processing Images: 100%|██████████| 3041/3041 [12:46<00:00,  3.97it/s]


--- Processing Complete ---
Positives: 15286
Hard Negatives: 21677
Backgrounds (Textured): 11721
Backgrounds (Flat): 17923
Discarded (Ambiguous): 76231
Skipped (Exact Duplicates): 17985
Skipped (Perceptual Duplicates): 540


## Create Synthetic Images

In [4]:
NUM_POSITIVES = 25_000
NUM_NEGATIVES = 75_000
generate_synthetic_data(
    config=config,
    num_positives=NUM_POSITIVES,
    num_negatives=NUM_NEGATIVES
)

Generator initialized: 145 High-Res images, 47 DTD categories.
--- Generator ---
Output: /home/nico/workspace/github.com/Nico-Sander/qr-code-detection-nico/dataset/patches
Positives: Found 15286 total. Generating 9714 new synthetic images.


Positives: 100%|██████████| 9714/9714 [13:52<00:00, 11.66it/s]


Negatives: Found 51321 total. Generating 23679 new synthetic images.


Negatives: 100%|██████████| 23679/23679 [32:33<00:00, 12.12it/s]  


## Split the Dataset into Train, Val and Test

In [9]:
split_dataset(config=config)

2026-02-05 16:14:23,296 - INFO - Starting Dataset Split: {'train': 0.9, 'val': 0.098, 'test': 0.002}
2026-02-05 16:14:23,584 - INFO - Processing Class: POSITIVE (25000 images)
2026-02-05 16:14:23,584 - INFO -   -> Train: 22500 | Val: 2450 | Test: 50
2026-02-05 16:14:25,920 - INFO - Processing Class: NEGATIVE (75000 images)          
2026-02-05 16:14:25,921 - INFO -   -> Train: 67500 | Val: 7350 | Test: 150
2026-02-05 16:14:28,969 - INFO - Cleaned up empty source folder: /home/nico/workspace/github.com/Nico-Sander/qr-code-detection-nico/dataset/patches/positive
2026-02-05 16:14:28,972 - INFO - Cleaned up empty source folder: /home/nico/workspace/github.com/Nico-Sander/qr-code-detection-nico/dataset/patches/negative
2026-02-05 16:14:28,972 - INFO - Dataset split complete!
2026-02-05 16:14:28,972 - INFO - Structure created at: /home/nico/workspace/github.com/Nico-Sander/qr-code-detection-nico/dataset/patches
